In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
 
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [3]:
import re
import sqlite3
import os

SQL_FILE = "eatdannys.sql"
DB_FILE = "eatdannys.sqlite"

# Start fresh
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

with open(SQL_FILE, "r", encoding="utf-8", errors="ignore") as f:
    sql = f.read()

# Tables we want
tables = ["meals", "categories"]

extracted_sql = []

for table in tables:
    # CREATE TABLE block
    create_pattern = rf"CREATE TABLE.*?`{table}`.*?\);"
    create_match = re.search(create_pattern, sql, flags=re.IGNORECASE | re.DOTALL)
    if create_match:
        extracted_sql.append(create_match.group())

    # INSERT statements
    insert_pattern = rf"INSERT INTO\s+`{table}`.*?;"
    inserts = re.findall(insert_pattern, sql, flags=re.IGNORECASE | re.DOTALL)
    extracted_sql.extend(inserts)

clean_sql = "\n".join(extracted_sql)


In [4]:
# MySQL → SQLite cleanup (focused)
import re

# Remove MySQL versioned comments
clean_sql = re.sub(r"/\*![\s\S]*?\*/", "", clean_sql)

# Remove CHARACTER SET ... COLLATE ...
clean_sql = re.sub(
    r"\sCHARACTER\s+SET\s+\w+\s+COLLATE\s+\w+",
    "",
    clean_sql,
    flags=re.IGNORECASE
)

# Remove standalone COLLATE ...
clean_sql = re.sub(
    r"\sCOLLATE\s+\w+",
    "",
    clean_sql,
    flags=re.IGNORECASE
)

# Remove table-level charset / collate
clean_sql = re.sub(
    r"DEFAULT\s+CHARSET=\w+",
    "",
    clean_sql,
    flags=re.IGNORECASE
)

clean_sql = re.sub(
    r"COLLATE=\w+",
    "",
    clean_sql,
    flags=re.IGNORECASE
)

# Other MySQL cleanup
clean_sql = re.sub(r"ENGINE=\w+", "", clean_sql, flags=re.IGNORECASE)
clean_sql = clean_sql.replace("AUTO_INCREMENT", "AUTOINCREMENT")
clean_sql = clean_sql.replace("UNSIGNED", "")
clean_sql = re.sub(r"\sDEFAULT\s+[^,\n]+", "", clean_sql, flags=re.IGNORECASE)

clean_sql = re.sub(
    r",?\s*CHECK\s*\(\s*json_valid\s*\([^)]+\)\s*\)",
    "",
    clean_sql,
    flags=re.IGNORECASE
)

clean_sql = re.sub(
    r"VALUES\s*\(([^)]*)\)",
    lambda m: m.group(0).replace("''", "NULL"),
    clean_sql,
    flags=re.DOTALL
)
 
clean_sql = clean_sql.replace("''", "NULL") 


In [5]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.executescript(clean_sql)
conn.commit()


OperationalError: table `cache` already exists

In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('cache',), ('cache_locks',), ('categories',), ('meals',)]

In [ ]:
cursor.execute("PRAGMA table_info(meals);")
cursor.fetchall()

In [7]:
cursor.execute("SELECT * FROM meals LIMIT 10;")
cursor.fetchall()


[(1,
  1,
  'Efo Riro with Fish',
  'Rich Nigerian spinach stew slow-cooked with tender fish, palm oil, and vibrant spices for an authentic, comforting flavor.',
  '[{"size": "2L", "price": 60}, {"size": "3L", "price": 75}, {"size": "6L", "price": 135}]',
  None,
  '2025-06-08 17:49:31'),
 (2,
  1,
  'Efo Riro with Assorted',
  'Hearty Nigerian spinach stew loaded with assorted meats, palm oil, and bold spices for a flavorful traditional experience.',
  '[{"size": "2L", "price": 60}, {"size": "3L", "price": 75}, {"size": "6L", "price": 135}]',
  None,
  None),
 (3,
  1,
  'Egusi Soup',
  'Creamy melon seed soup packed with vegetables, meats, and spices – a classic Nigerian favorite.',
  '[{"size": "2L", "price": 60}, {"size": "3L", "price": 85}, {"size": "6L", "price": 155}]',
  None,
  None),
 (4,
  1,
  'Oha Soup',
  'Aromatic Eastern Nigerian soup with oha leaves, thickened with cocoyam and rich meats.',
  '[{"size": "2L", "price": 95}, {"size": "3L", "price": 140}, {"size": "6L", "

In [9]:
def get_meal_price(meal_name):
    print(f"DATABASE TOOL CALLED: Getting price for {meal_name}", flush=True)
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT prices FROM meals WHERE name = ?', (meal_name,))
        result = cursor.fetchone()
        return f"Meal price for {meal_name} is ${result[0]}" if result else "No price data available for this meal"

In [10]:
def get_meal_category(meal_name):
    print(f"DATABASE TOOL CALLED: Getting category for {meal_name}", flush=True)
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT categories.name FROM meals JOIN categories ON meals.category_id = categories.id WHERE meals.name = ?', (meal_name,))
        result = cursor.fetchone()
        return f"Meal category for {meal_name} is {result[0]}" if result else "No category data available for this meal"

In [14]:
def get_meals_under_a_category(category_name):
    print(f"DATABASE TOOL CALLED: Getting meals under {category_name}", flush=True)
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()  
        cursor.execute('SELECT meals.name FROM meals JOIN categories ON meals.category_id = categories.id WHERE categories.name = ?', (category_name,))
        results = cursor.fetchall()
        return f"Meals under {category_name} are {', '.join([r[0] for r in results])}" if results else "No meals found under this category"

In [16]:
get_meals_under_a_category("Soup")

DATABASE TOOL CALLED: Getting meals under Soup


'Meals under Soup are Assorted Pepper Soup, Assorted Pepper Soup, Assorted Stew, Assorted Stew, Banga Soup, Banga Soup, Beef Stew, Beef Stew, Bitter Leaf Soup, Bitter Leaf Soup, Catfish Pepper Soup, Catfish Pepper Soup, Chicken Stew, Chicken Stew, Cow Leg Pepper Soup, Cow Leg Pepper Soup, Efo Riro with Assorted, Efo Riro with Assorted, Efo Riro with Fish, Efo Riro with Fish, Egusi Soup, Egusi Soup, Ewedu, Ewedu, Fish Stew, Fish Stew, Gbegiri (Bean Soup), Gbegiri (Bean Soup), Goat Meat Stew, Goat Meat Stew, Ofada/Avamase (Red), Ofada/Avamase (Red), Ofada/Ayamase (Green), Ofada/Ayamase (Green), Ogbono Soup, Ogbono Soup, Oha Soup, Oha Soup, Seafood Okro, Seafood Okro, Turkey Stew, Turkey Stew'

In [12]:
get_meal_category("Efo Riro with Fish")

DATABASE TOOL CALLED: Getting category for Efo Riro with Fish


'Meal category for Efo Riro with Fish is Soup'

In [11]:
get_meal_price("Efo Riro with Fish")

DATABASE TOOL CALLED: Getting price for Efo Riro with Fish


'Meal price for Efo Riro with Fish is $[{"size": "2L", "price": 60}, {"size": "3L", "price": 75}, {"size": "6L", "price": 135}]'

In [17]:
 price_function = {
  'name': 'get_meal_price',
  'description': 'Get the price of a meal. Call this whenever you need to know the price of a meal, for example when a customer asks "How much is this meal?"',
  'parameters': {
    'type': 'object',
    'properties': {
      'meal_name': {
        'type': 'string',
        'description': 'The name of the meal that the customer wants to purchase'
      }
    },
    'required': ['meal_name'],
    'additionalProperties': False
  }
 }

In [18]:
tools = [
    {"type": "function", "function": price_function}
]